# Radiology AI popularity and the pediatric subset

Interactive exploration of the data collected by `scripts/run_all.py`.
Run the collectors first (or `python scripts/run_all.py --quick`) so that
`data/processed/` is populated. This notebook only reads cached results, so it
does not hit the network.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from pedrad_ai import analysis, config

proc = config.PROCESSED_DIR
counts = json.loads((proc / 'pubmed_yearly_counts.json').read_text())
df = pd.DataFrame(analysis.fractions_over_time(counts)).set_index('year')
df

## Growth of radiology AI and its share of all radiology

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.bar(df.index, df['radiology_ai'], color='#3b6ea5', alpha=0.8, label='Radiology-AI papers')
ax1.set_ylabel('Radiology-AI publications')
ax2 = ax1.twinx()
ax2.plot(df.index, df['ai_share_of_radiology'] * 100, 'o-', color='#c44e52')
ax2.set_ylabel('AI share of all radiology (%)')
ax1.set_xlabel('Year'); plt.title('Growth of radiology AI'); plt.show()

## Pediatric share of radiology AI over time

In [ ]:
ax = (df['pediatric_share_of_radiology_ai'] * 100).plot(marker='s', figsize=(9, 4.5), color='#55a868')
ax.set_ylabel('Pediatric share of radiology AI (%)'); ax.set_xlabel('Year')
ax.set_title('Pediatric fraction of radiology-AI publications'); plt.show()

## Modality and task breakdown

In [ ]:
for prefix in ('modality', 'task', 'ped_modality'):
    bd = pd.DataFrame(analysis.breakdown_table(counts, prefix))
    if not bd.empty:
        print(f'\n=== {prefix} ===')
        display(bd)

## Biggest players: most-cited papers and most-starred tools

In [ ]:
for f in ('top_papers_radiology_ai.json', 'top_papers_pediatric_radiology_ai.json'):
    p = proc / f
    if p.exists():
        papers = json.loads(p.read_text())
        print(f'\n=== {f} ===')
        display(pd.DataFrame(papers)[['citation_count', 'year', 'title', 'venue']].head(15))

In [ ]:
lb = proc / 'github_leaderboard.csv'
if lb.exists():
    display(pd.read_csv(lb).head(20))